<a href="https://colab.research.google.com/github/deiteris/voice-changer/blob/master-custom/Colab_RealtimeVoiceChanger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### [W-okada's Deiteris' Fork Realtime Voice Changer](https://github.com/deiteris/voice-changer) | **Google Colab**

---

## **READ ME - VERY IMPORTANT**

You can use the following settings for optimal results:

Best performance (with good quality): `f0: fcpe | Chunk: 80.0ms or higher | Extra: 5s`<br>
Best quality: `f0: rmvpe | Chunk: 112.0ms or higher | Extra: 5s`<br>
**Don't forget to select your Colab GPU in the GPU field (<b>Tesla T4</b>, for free users)*

You can tune `Chunk` for lower/higher delay and `Extra` for better quality.

<br><br>

---

### Always use Colab GPU (**VERY VERY VERY IMPORTANT!**)
You need to use a Colab GPU so the Voice Changer can work faster and better\
Use the menu above and click on **Runtime** » **Change runtime** » **Hardware acceleration** to select a GPU (**T4 is the free one**)

---

# **Credits and Support**
Original W-okada Realtime Voice Changer by [w-okada](https://github.com/w-okada)<br>
Original instructions by [Hina](https://github.com/HinaBl)<br>

Need help? [AI Hub Discord](https://discord.gg/aihub) » ***#help-w-okada***

---

In [ ]:
#=================Updated=================
# @title **[1]** Clone repository and install dependencies
# @markdown This first step will download the latest version of Voice Changer and install the dependencies. **It can take some time to complete.**
import os
import time
from google.colab import drive

#@markdown ---
# @title **[Optional]** Connect to Google Drive
# @markdown Using Google Drive will automatically save your uploaded models for later use. Make sure you have sufficient amount of space on your Google Drive.

%cd /content

Use_Drive=True #@param {type:"boolean"}

!pip install colorama python-dotenv "pyngrok>=9,<10" --quiet
from colorama import Fore, Style

import re
import subprocess
import requests

Release_Repository = "JohnRealNoob/voice-changer" #@param {type:"string"}
ARCHIVE = "voice-changer-linux-amd64-cuda.tar.gz"
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", Release_Repository):
    raise ValueError("Release_Repository must have the form owner/repository")
response = requests.get(f"https://api.github.com/repos/{Release_Repository}/releases/latest", timeout=30)
if response.status_code == 404:
    raise RuntimeError(f"No published release found for {Release_Repository}. Publish a release or select another repository.")
response.raise_for_status()
release_info = response.json()

from pathlib import Path
from tempfile import TemporaryDirectory, mkdtemp
import shutil

def download_and_unpack(assets, staging, archive):
    matches = {asset["name"]: asset for asset in assets
               if asset["name"] == archive or re.fullmatch(re.escape(archive) + r"\.[a-z]{2}", asset["name"])}
    names = [archive] if archive in matches else sorted(matches)
    if not names or (names != [archive] and names != [
        archive + "." + chr(97 + i // 26) + chr(97 + i % 26) for i in range(len(names))
    ]):
        raise RuntimeError("No complete CUDA archive found in the selected release.")
    with subprocess.Popen(["tar", "-xzf", "-", "-C", str(staging)], stdin=subprocess.PIPE) as extract:
        with extract.stdin:
            for name in names:
                with requests.get(matches[name]["browser_download_url"], stream=True, timeout=60) as download:
                    download.raise_for_status()
                    for chunk in download.iter_content(chunk_size=1024 * 1024):
                        extract.stdin.write(chunk)
        if extract.wait() != 0:
            raise RuntimeError("Failed to extract the release archive.")


def install_runtime(assets, install, executable, archive):
    with TemporaryDirectory(prefix=".vc-download-", dir=install.parent) as staging:
        source = Path(staging) / "MMVCServerSIO"
        download_and_unpack(assets, source.parent, archive)
        binary = source / "MMVCServerSIO"
        internal = source / "_internal"
        if not binary.is_file() or not os.access(binary, os.X_OK) or not (internal / "version.txt").is_file():
            raise RuntimeError("The downloaded release is missing its executable or version file.")

        install.mkdir(exist_ok=True)
        backup = Path(mkdtemp(prefix=".vc-backup-", dir=install.parent))
        saved, replaced = [], []
        try:
            for incoming, name in ((binary, executable), (internal, "_internal")):
                destination = install / name
                if destination.exists() or destination.is_symlink():
                    destination.rename(backup / name)
                    saved.append(name)
                incoming.rename(destination)
                replaced.append((destination, incoming))
        except (OSError, KeyboardInterrupt):
            for destination, incoming in reversed(replaced):
                destination.rename(incoming)
            for name in reversed(saved):
                (backup / name).rename(install / name)
            backup.rmdir()
            raise
        else:
            shutil.rmtree(backup)


REAL1 = "voice-changer"
install = Path("/content/ML_Program")
install_runtime(release_info["assets"], install, "ML_Program", ARCHIVE)

if Use_Drive:
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive') / REAL1 / 'server'
    for name in ('model_dir', 'pretrain'):
        target = drive_dir / name
        target.mkdir(parents=True, exist_ok=True)
        link = install / name
        if link.is_symlink() and link.resolve() == target.resolve():
            continue
        if link.exists() or link.is_symlink():
            print(f'Keeping {link}. Move it to {target} before linking Google Drive.')
            continue
        link.symlink_to(target, target_is_directory=True)

print(f"{Fore.GREEN}> Successfully downloaded and unpacked the binary!{Style.RESET_ALL}")

print(f"{Fore.CYAN}> Installing libportaudio2...{Style.RESET_ALL}")
!apt-get -y install libportaudio2 -qq

In [ ]:
#=================Updated=================
# @title **[2]** Set server configuration
# @markdown This cell will set the server configuration.

%cd /content/ML_Program

from dotenv import set_key

set_key('.env', "SAMPLE_MODE", "")

Ready = True

In [ ]:
# @title Start an authenticated server using ngrok
# @markdown Enter your ngrok token and a separate visitor username and password when prompted.
# @markdown Visitors must enter those credentials in their browser. Keep them private.
# @markdown HRZN is disabled because visitor access controls have not been verified.
from getpass import getpass
import json, socket, threading, time
from IPython.display import clear_output
from pyngrok import conf, ngrok
from dotenv import set_key

if not globals().get('Ready', False):
    raise RuntimeError("Run the setup and configuration cells first.")
ngrok_token = getpass('ngrok auth token: ').strip()
visitor_user = input('Visitor username: ').strip()
visitor_password = getpass('Visitor password: ')
if not ngrok_token or not visitor_user or ':' in visitor_user or not visitor_password:
    raise ValueError('A token, username (without colon), and password are required.')
config = conf.PyngrokConfig(auth_token=ngrok_token, config_version='3')
policy = {'on_http_request': [{'actions': [{'type': 'basic-auth', 'config': {'credentials': [f'{visitor_user}:{visitor_password}']}}]}]}
PORT = 18888
connection = ngrok.connect(PORT, traffic_policy=policy, pyngrok_config=config)
public_url = connection.public_url
try:
    set_key('.env', 'ALLOWED_ORIGINS', json.dumps([public_url]))
    def wait_for_server():
        while True:
            time.sleep(0.5)
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
                if sock.connect_ex(('127.0.0.1', PORT)) == 0:
                    break
        print('Server ready:', public_url)
    threading.Thread(target=wait_for_server, daemon=True).start()
    !./ML_Program
finally:
    ngrok.disconnect(public_url, pyngrok_config=config)
    del ngrok_token, visitor_password, policy
